# Predictor de Tráfico

Este cuaderno recoge la versión actual del proyecto: carga de datos desde la API, preparación del conjunto, entrenamiento del modelo, mapa de estaciones y la interfaz final en Streamlit.

La intención es dejar aquí una guía clara del flujo completo, reutilizando los mismos módulos que usa la aplicación para no repetir código.

## Carga de datos

Primero cargamos el dataset principal del Cabildo y dejamos fuera de aquí la lógica de la API, que ya vive en su propio módulo.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from api_trafico import cargar_datos_api

URL_BASE = "https://datos.tenerife.es/ckan"
ID_PAQUETE = "d02ccf54-aca2-4a82-9ef0-fbd285b5f401"

dataset_crudo = cargar_datos_api(URL_BASE, ID_PAQUETE)
dataset_crudo.head()

## Preprocesamiento

Aquí se preparan las variables que necesita el modelo: fecha, estación, sentido, hora y día de la semana. El objetivo es quedarnos con una base limpia y fácil de reutilizar en la app.

In [ ]:
from preprocesamiento import preprocesar_datos

X, y, le_estacion, df_agrupado = preprocesar_datos(dataset_crudo)

df_agrupado.head()

## Entrenamiento del modelo

El modelo que usa la app se entrena con estas mismas variables. En el cuaderno se muestra la idea general, pero la implementación final vive en el módulo `modelo.py` para que la aplicación sea más fácil de mantener.

In [ ]:
from modelo import entrenar_modelo

modelo_entrenado = entrenar_modelo(X, y)
modelo_entrenado

## Interfaz de predicción

La aplicación final se construye con Streamlit. Incluye el selector de estación, el mapa con las ubicaciones oficiales y el formulario donde se calcula la predicción.

In [ ]:
from api_estaciones import cargar_estaciones_aforo
from preprocesamiento import preparar_entrada_usuario, estimar_posibilidad_atasco

ID_PAQUETE_ESTACIONES = "a114f82b-bb62-47e0-94bb-bebb484514a7"
estaciones_geo = cargar_estaciones_aforo(URL_BASE, ID_PAQUETE_ESTACIONES)

estaciones_geo[["estacion_id", "estacion_nombre", "estacion_latitud", "estacion_longitud"]].head()

## Resumen

Con esto queda recogido el flujo completo: carga desde la API, preparación, modelo, mapa y predicción. La app usa la misma lógica, así que el cuaderno sirve como guía sin duplicar la implementación.